In [1]:
import sys

print(sys.executable)
print(sys.version)

/home/ayxiao227/dft_ml_project/.venv/bin/python
3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]


In [41]:
import pubchempy as pcp
import pandas as pd
import regex as re


In [68]:
properties = ["IUPACName", "MolecularFormula", "CanonicalSMILES", "Charge", "HeavyAtomCount"]

In [67]:
molecules = ["methane", "ethanol", "acetone", "ammonia"]

In [69]:
constraints = {
    "max_molecular_weight": 150,
    "max_heavy_atoms": 10,
    "allowed_elements": {"C", "H", "O", "N"},
    "neutral_only": True
}

Formula to elements

In [70]:
def get_elements_from_smiles_regex(smiles):
    # bracket atoms: [ ... ]  |  two-letter organic subset: Br, Cl  |  single-letter (incl. aromatic)
    token_re = re.compile(r'\[([^\]]+)\]|(Br|Cl|[BCNOPSFI]|[bcnops])')

    elements = set()
    for m in token_re.finditer(smiles):
        bracket_content, plain = m.group(1), m.group(2)
        if bracket_content is not None:
            # inside brackets: optional isotope digits, then the element symbol
            mm = re.match(r'^\d*([A-Za-z][a-z]?)', bracket_content)
            if mm:
                sym = mm.group(1)
                elements.add(sym.capitalize())
        else:
            elements.add(plain.capitalize())
    return elements


In [57]:
print(get_elements_from_smiles_regex("ccccconn"))

{'N', 'O', 'C'}


In [62]:
print(get_elements_from_smiles_regex("ccccconn").issubset(constraints["allowed_elements"]))

True


In [72]:
print(properties)
print(molecules)

['IUPACName', 'MolecularFormula', 'CanonicalSMILES', 'Charge', 'HeavyAtomCount']
['methane', 'ethanol', 'acetone', 'ammonia']


In [76]:
result = pcp.get_properties(properties, "methane", "name")

In [77]:
print(result)

[{'CID': 297, 'MolecularFormula': 'CH4', 'ConnectivitySMILES': 'C', 'IUPACName': 'methane', 'Charge': 0, 'HeavyAtomCount': 1}]


In [ ]:
data = pd.DataFrame(columns = properties)
valid_data = pd.DataFrame(columns = ["CID", "MolecularFormula", "ConnectivitySMILES", "IUPACName"])
for name in molecules:
    # Using as_dataframe=False prevents the KeyError if a molecule isn't found
    #result = pcp.get_cids(name, "name")
    result = pcp.get_properties(properties, name, "name", as_dataframe = True)
    if not result.empty:
        data.append(result)
        #no_num_formula = "".join([char for char in result[3] if not char.isdigit()])
        #for word in no_num_formula:
            #if word not in constraints["allowed_elements"]:
                #continue
        #if get_elements_from_smiles_regex(result[2]).issubset(constraints["allowed_elements"]) and result[3] == 0 and result[4] == 0:
            #valid_data
        elements_ok = data["ConnectivitySMILES"].apply(get_elements_from_smiles_regex).apply(lambda els: els.issubset(constraints["allowed_elements"]))
        mask = elements_ok & (data["Charge"] == 0) & (data["HeavyAtomCount"] < 10)
        valid_df = data[mask]
    else:
        print(f"Warning: Could not find '{name}' in PubChem.")

AttributeError: 'DataFrame' object has no attribute 'append'

In [74]:
print(molecules)
print(valid_molecules)

['methane', 'ethanol', 'acetone', 'ammonia', {'CID': 297, 'MolecularFormula': 'CH4', 'ConnectivitySMILES': 'C', 'IUPACName': 'methane', 'Charge': 0, 'HeavyAtomCount': 1}, {'CID': 702, 'MolecularFormula': 'C2H6O', 'ConnectivitySMILES': 'CCO', 'IUPACName': 'ethanol', 'Charge': 0, 'HeavyAtomCount': 3}, {'CID': 180, 'MolecularFormula': 'C3H6O', 'ConnectivitySMILES': 'CC(=O)C', 'IUPACName': 'propan-2-one', 'Charge': 0, 'HeavyAtomCount': 4}, {'CID': 222, 'MolecularFormula': 'H3N', 'ConnectivitySMILES': 'N', 'IUPACName': 'azane', 'Charge': 0, 'HeavyAtomCount': 1}]
[]
